<a href="https://colab.research.google.com/github/emmners/210final/blob/main/DS340_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# import PyTorch
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# data processing imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# computer vision
import cv2

# evaluation imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("All libraries imported successfully!")

# check GPU as device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

All libraries imported successfully!
Using device: cuda


In [2]:
import zipfile
import os

# location of uploaded kaggle dataset ZIP file
zip_path = "/content/archive.zip"

# extract the dataset
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/fer2013")

print("dataset successfully extracted")

FileNotFoundError: [Errno 2] No such file or directory: '/content/archive.zip'

In [ ]:
# checking that it worked
# should extract a train and test folder
import os

print(os.listdir("/content/fer2013"))

In [ ]:
# looking for if there is class imbalance (there is)
train_path = "/content/fer2013/train"

for emotion in sorted(os.listdir(train_path)):
    folder = os.path.join(train_path, emotion)

    if os.path.isdir(folder):
        print(emotion, ":", len(os.listdir(folder)), "images")

In [ ]:
# plot a sample image of each emotion
import matplotlib.pyplot as plt
import cv2
import os
import random

emotions = [
    "angry", "disgust", "fear",
    "happy", "neutral", "sad", "surprise"
]

train_path = "/content/fer2013/train"

fig, axes = plt.subplots(1, 7, figsize=(18, 4))

for i, emotion in enumerate(emotions):

    folder = os.path.join(train_path, emotion)

    image_name = random.choice(os.listdir(folder))
    image_path = os.path.join(folder, image_name)

    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)

    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(emotion.capitalize())
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# plot distribution of images to see the imbalance
counts = {}

for emotion in emotions:
    folder = os.path.join(train_path, emotion)
    counts[emotion] = len(os.listdir(folder))

plt.figure(figsize=(10, 5))

plt.bar(counts.keys(), counts.values(), color = "skyblue")

plt.title("FER-2013 Training Dataset Distribution")
plt.xlabel("Facial Expression")
plt.ylabel("Number of Images")

plt.xticks(rotation = 30)
plt.show()

In [ ]:
# import preprocessing libraries
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# define image transformations
# transformations for training images
# receive random flips and rotations to help model generalize
train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels = 3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# transformations for testing images
# remain unchanged to evaluate model fairly
test_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels = 3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# load tranformed images using ImageFolder
# note: ImageFolder assigns class numbers alphabetically
train_path = "/content/fer2013/train"
test_path = "/content/fer2013/test"

train_dataset = datasets.ImageFolder(root = train_path, transform = train_transform)
test_dataset = datasets.ImageFolder(root = test_path, transform = test_transform)

print("Train dataset size:", len(train_dataset))
print("Test dataset size:", len(test_dataset))
print("Class name: ", train_dataset.classes)

In [ ]:
# now DataLoader to divide the dataset into smaller batches
# helps model process images efficiently
train_loader = DataLoader(
    train_dataset,
    batch_size = 32,
    shuffle = True,
    num_workers = 2
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 32,
    shuffle = False,
    num_workers = 2
)

print("DataLoaders created successfully")

In [ ]:
# sanity check to make sure preprocessing worked
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)

In [ ]:
# visualize a preprocessed image
# check for: grayscale, square shape
image, label = train_dataset[0]

# undo normalization for visualization
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

image = image * std + mean
image = image.permute(1, 2, 0).numpy()

plt.imshow(image)
plt.title(train_dataset.classes[label])
plt.axis("off")
plt.show()

In [ ]:
# building a first simple CNN baseline model using PyTorch
# performance benchmark to compare against ResNet-18 later

# Conv2d: learns visual features from facial images
# ReLU: adds nonlinearity so the model can learn complex patterns
# MaxPool2d: reduces image dimensions
# Dropout: brain damage neuronrs to help reduce overfitting
# linear: produces the final seven expression scores

import torch
import torch.nn as nn
import torch.nn.functional as F

class BaselineCNN(nn.Module):

    def __init__(self):
        super(BaselineCNN, self).__init__()

        # first convolutional layer
        self.conv1 = nn.Conv2d(
            3, 32, kernel_size = 3, padding = 1
        )

        # second convolutional layer
        self.conv2 = nn.Conv2d(
            32, 64, kernel_size = 3, padding = 1
        )

        # Third convolutional layer
        self.conv3 = nn.Conv2d(
            64, 128, kernel_size = 3, padding = 1
        )

        self.pool = nn.MaxPool2d(2, 2)

        # Fully connected layers
        self.fc1 = nn.Linear(128 * 28 * 28, 256)
        self.fc2 = nn.Linear(256, 7)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):

        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        x = torch.flatten(x, 1)

        x = F.relu(self.fc1(x))
        x = self.dropout(x)

        x = self.fc2(x)

        return x

In [ ]:
# initialize model
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BaselineCNN().to(device)

print(model)
print("Using device:", device)

In [ ]:
# define loss function and optimizer
# goal: minimize classification errors
# since we have class imbalance, we'll use a weighted loss so that minorities get more importance

import numpy as np

# some arithmetic

# count training images in each class
class_counts = np.bincount(train_dataset.targets)

# calculate class weights
class_weights = len(train_dataset) / (
    len(class_counts) * class_counts
)

class_weights = torch.tensor(
    class_weights,
    dtype = torch.float32
).to(device)

# cross entropy loss function
criterion = nn.CrossEntropyLoss(
    weight = class_weights
)

# Adam optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr = 0.001
)

print("Class weights:", class_weights)
print("Loss function and optimizer ready")

In [ ]:
# train CNN with 10 epochs
# note: this only tracks training performance
# will use a separate validation set to monitor overfitting before we choose a final model
epochs = 10

train_losses = []
train_accuracies = []

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # rest gradients
        optimizer.zero_grad()

        # forward pass
        outputs = model(images)

        # calcuate loss
        loss = criterion(outputs, labels)

        # backprop
        loss.backward()

        # update model parameters
        optimizer.step()

        running_loss += loss.item()

        # calculate accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total

    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {epoch_loss:.4f} "
        f"Accuracy: {epoch_accuracy:.2f}%"
    )

In [ ]:
# plot training results
# accuracy is not really improving, so model is barely learning
# probably because its a large fully connected layer or something with the class weighting

# main fixes
# keep images at original 48 × 48 resolution
# use one grayscale channel instead of three.
# use adaptive average pooling instead of a large fully connected layer

plt.figure(figsize=(12, 4))

# loss graph
plt.subplot(1, 2, 1)
plt.plot(train_losses, marker='o')
plt.title("CNN Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

# accuracy graph
plt.subplot(1, 2, 2)
plt.plot(train_accuracies, marker='o', color='green')
plt.title("CNN Training Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.tight_layout()
plt.show()

In [ ]:
# create new preprocessing transformations
# creating a validation set from the training data
# so we can monitor learning without exposing model to test data too early

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

baseline_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels = 1),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

baseline_val_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels = 1),
    transforms.ToTensor()
])

train_path = "/content/fer2013/train"

# separate datasets so validation images aren't augmented
train_full = datasets.ImageFolder(
    train_path,
    transform = baseline_train_transform
)

val_full = datasets.ImageFolder(
    train_path,
    transform = baseline_val_transform
)

# 90/10 training-validation split
indices = np.arange(len(train_full))

train_indices, val_indices = train_test_split(
    indices,
    test_size = 0.10,
    random_state = 42,
    stratify=train_full.targets
)

train_subset = Subset(train_full, train_indices)
val_subset = Subset(val_full, val_indices)

train_loader = DataLoader(
    train_subset,
    batch_size = 64,
    shuffle = True,
    num_workers = 2
)

val_loader = DataLoader(
    val_subset,
    batch_size = 64,
    shuffle = False,
    num_workers = 2
)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))

In [ ]:
# create improved CNN
class BaselineCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(1, 32, 3, padding = 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding = 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding = 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 7)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BaselineCNN().to(device)

print(model)

In [ ]:
# a less aggressive class weighting strategy
# how: using square roots of weights instead of reverse inverse frequency
class_counts = np.bincount(
    np.array(train_full.targets)[train_indices],
    minlength = 7
)

class_weights = np.sqrt(
    class_counts.sum() / (7 * class_counts)
)

class_weights = torch.tensor(
    class_weights,
    dtype = torch.float32
).to(device)

criterion = nn.CrossEntropyLoss(
    weight = class_weights
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr = 0.001,
    weight_decay = 0.0001
)

print("Class weights:", class_weights)

In [ ]:
# retrain CNN with 10 epochs again
epochs = 10

train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

for epoch in range(epochs):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        predicted = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total

    # validation
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predicted = outputs.argmax(dim=1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

In [ ]:
# plot results again cus there are fluctuations
# much better results, and technically do have a useable baseline against ResNet-18
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, 11), train_losses, marker='o', label='Training')
plt.plot(range(1, 11), val_losses, marker='o', label='Validation')
plt.title("CNN Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid()

plt.subplot(1, 2, 2)
plt.plot(range(1, 11), train_accuracies, marker='o', label='Training')
plt.plot(range(1, 11), val_accuracies, marker='o', label='Validation')
plt.title("CNN Accuracy Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid()

plt.tight_layout()
plt.show()

In [ ]:
# now we evaluate this baseline CNN model on the test data
# note: the original test dataset uses old 224 x 224 preprocessing
# our new CNN expectes 48 x 48 grayscale images, so first recreate test dataset

test_transform_baseline = transforms.Compose([
    transforms.Grayscale(num_output_channels = 1),
    transforms.ToTensor()
])

test_dataset_baseline = datasets.ImageFolder(
    root="/content/fer2013/test",
    transform = test_transform_baseline
)

test_loader_baseline = DataLoader(
    test_dataset_baseline,
    batch_size = 64,
    shuffle = False,
    num_workers = 2
)

print("Test images:", len(test_dataset_baseline))

In [ ]:
# now evaluate and visualize a confustion matrix since some expressions look similar
# results: looks like the model is still favoring some classes over others

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix, accuracy_score

model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader_baseline:

        images = images.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

print("Baseline CNN Test Accuracy:",
      round(test_accuracy * 100, 2), "%")

print(classification_report(
    all_labels,
    all_predictions,
    target_names = test_dataset_baseline.classes,
    zero_division = 0
))

In [ ]:
# prepare ResNet-18 datasets
# using 224 x 224 images just like the pretrained model expects
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import numpy as np

# using pretrained ImageNet weights
# ImageNet normalization
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# training transformations
resnet_train_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# validation and testing transformations
resnet_eval_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# create separate datasets
train_full_resnet = datasets.ImageFolder(
    "/content/fer2013/train",
    transform=resnet_train_transform
)

val_full_resnet = datasets.ImageFolder(
    "/content/fer2013/train",
    transform=resnet_eval_transform
)

# split
indices = np.arange(len(train_full_resnet))

train_indices, val_indices = train_test_split(
    indices,
    test_size = 0.10,
    random_state = 42,
    stratify = train_full_resnet.targets
)

train_resnet = Subset(train_full_resnet, train_indices)
val_resnet = Subset(val_full_resnet, val_indices)

test_resnet = datasets.ImageFolder(
    "/content/fer2013/test",
    transform=resnet_eval_transform
)

# DataLoaders
train_loader_resnet = DataLoader(
    train_resnet,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader_resnet = DataLoader(
    val_resnet,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader_resnet = DataLoader(
    test_resnet,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Training:", len(train_resnet))
print("Validation:", len(val_resnet))
print("Testing:", len(test_resnet))

In [ ]:
# load pretrained ResNet-18 model
import torch
import torch.nn as nn
from torchvision import models

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# load pretrained ResNet-18
weights = models.ResNet18_Weights.DEFAULT

model_resnet = models.resnet18(weights=weights)

# replace the final layer with seven expression classes (transfer learning hehe)
num_features = model_resnet.fc.in_features

model_resnet.fc = nn.Linear(
    num_features, 7
)

model_resnet = model_resnet.to(device)

print("ResNet-18 loaded successfully")
print("Output classes:", model_resnet.fc.out_features)

In [ ]:
# train new classification layer first

# freeze pretrained layers
for param in model_resnet.parameters():
    param.requires_grad = False

# train only the final classification layer
for param in model_resnet.fc.parameters():
    param.requires_grad = True

# calculate class weights
class_counts = np.bincount(
    np.array(train_full_resnet.targets)[train_indices],
    minlength = 7
)

class_weights_resnet = np.sqrt(
    class_counts.sum() / (7 * class_counts)
)

class_weights_resnet = torch.tensor(
    class_weights_resnet,
    dtype=torch.float32
).to(device)

criterion_resnet = nn.CrossEntropyLoss(
    weight=class_weights_resnet
)

optimizer_resnet = torch.optim.Adam(
    model_resnet.fc.parameters(),
    lr=0.001
)

print("ResNet-18 classification layer ready")

In [ ]:
# train classification layer but with just 3 epochs
# good results: accuracy is already increasing and loss is decreasing
for epoch in range(3):

    model_resnet.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader_resnet:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_resnet.zero_grad()

        outputs = model_resnet(images)

        loss = criterion_resnet(outputs, labels)

        loss.backward()
        optimizer_resnet.step()

        running_loss += loss.item()

        predicted = outputs.argmax(dim = 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(
        f"Epoch [{epoch+1}/3] "
        f"Loss: {running_loss / len(train_loader_resnet):.4f} "
        f"Accuracy: {100 * correct / total:.2f}%"
    )

In [ ]:
# finetune ResNet-18
# allows pretrained model to adapt some of its learned features to facial expression recognition

# unfreeze final residual block
for param in model_resnet.layer4.parameters():
    param.requires_grad = True

# keep final classification layer trainable
for param in model_resnet.fc.parameters():
    param.requires_grad = True

# use different learning rates
optimizer_resnet = torch.optim.AdamW([
    {
        "params": model_resnet.layer4.parameters(),
        "lr": 0.0001
    },
    {
        "params": model_resnet.fc.parameters(),
        "lr": 0.0005
    }
], weight_decay = 0.0001)

print("ResNet-18 is ready for fine-tuning")

In [ ]:
# train for another 10 epochs and track training and validation performance
# results: training accuracy increases while validation accuracy levels off
# much better than our baseline CNN model
epochs = 10

resnet_train_losses = []
resnet_train_accs = []
resnet_val_losses = []
resnet_val_accs = []

for epoch in range(epochs):

    model_resnet.train()

    # keep frozen feature layers in evaluation mode
    # prevents pretrained stats from changing
    for name, module in model_resnet.named_modules():
        if isinstance(module, nn.BatchNorm2d):
            if not any(
                p.requires_grad for p in module.parameters()
            ):
                module.eval()

    running_loss = 0
    correct = 0
    total = 0

    # training
    for images, labels in train_loader_resnet:

        images = images.to(device)
        labels = labels.to(device)

        optimizer_resnet.zero_grad()

        outputs = model_resnet(images)
        loss = criterion_resnet(outputs, labels)

        loss.backward()
        optimizer_resnet.step()

        running_loss += loss.item()

        predicted = outputs.argmax(dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader_resnet)
    train_acc = 100 * correct / total

    # validation
    model_resnet.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader_resnet:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model_resnet(images)

            loss = criterion_resnet(outputs, labels)

            val_loss += loss.item()

            predicted = outputs.argmax(dim=1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss /= len(val_loader_resnet)
    val_acc = 100 * val_correct / val_total

    resnet_train_losses.append(train_loss)
    resnet_train_accs.append(train_acc)
    resnet_val_losses.append(val_loss)
    resnet_val_accs.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

In [ ]:
# save this ResNet-18 model as a checkpoint
torch.save(
    model_resnet.state_dict(),
    "/content/resnet18_epoch10.pth"
)

print("ResNet-18 saved")

In [ ]:
# evaluate ResNet-18 on test dataset now
# and generate a confusion matrix
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix
)

model_resnet.eval()

all_predictions_resnet = []
all_labels_resnet = []

with torch.no_grad():

    for images, labels in test_loader_resnet:

        images = images.to(device)

        outputs = model_resnet(images)

        predictions = outputs.argmax(dim=1)

        all_predictions_resnet.extend(
            predictions.cpu().numpy()
        )

        all_labels_resnet.extend(labels.numpy())

resnet_test_accuracy = accuracy_score(
    all_labels_resnet,
    all_predictions_resnet
)

print(
    "ResNet-18 Test Accuracy:",
    round(resnet_test_accuracy * 100, 2),
    "%"
)

print(classification_report(
    all_labels_resnet,
    all_predictions_resnet,
    target_names = test_resnet.classes,
    zero_division = 0
))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    all_labels_resnet,
    all_predictions_resnet
)

plt.figure(figsize=(9, 7))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_resnet.classes,
    yticklabels=test_resnet.classes
)

plt.title("ResNet-18 Confusion Matrix")
plt.xlabel("Predicted Expression")
plt.ylabel("Actual Expression")

plt.show()

In [ ]:
torch.save(
    model_resnet.state_dict(),
    "/content/resnet18_fer2013.pth"
)

print("model saved successfully")

In [ ]:
import json

class_names = train_full_resnet.classes

with open("/content/class_names.json", "w") as f:
    json.dump(class_names, f)

print(class_names)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# main dataset location
DATASET_PATH = "/content/drive/MyDrive/cat-emotion-classifier/dataset"

# the seven emotions
emotions = [
    "angry",
    "disgust",
    "fear",
    "happy",
    "neutral",
    "sad",
    "surprise"
]

# create folders
for emotion in emotions:
    os.makedirs(os.path.join(DATASET_PATH, emotion), exist_ok=True)

print("Dataset folders created!")